# # User Model Definition

In [ ]:
# `load_model` runs once at server start; `infer` runs per request.
# Imports used inside these methods must live inside them (the class
# is cloudpickled when shipped).
#   load_model(context)  — context["worker_index"], context["num_workers"]
#   infer(**kwargs)      — kwargs come from the request JSON body; the
#                          return value must be JSON-serializable.
from nubison_model import NubisonModel, ModelContext


class UserModel(NubisonModel):
    def load_model(self, context: ModelContext) -> None:
        import pickle
        with open("src/weights.pkl", "rb") as f:
            self.estimator = pickle.load(f)

    def infer(self, sepal_length: float, sepal_width: float,
              petal_length: float, petal_width: float) -> dict:
        import pandas as pd
        x = pd.DataFrame(
            [[sepal_length, sepal_width, petal_length, petal_width]],
            columns=["sepal_length", "sepal_width", "petal_length", "petal_width"],
        )
        return {"target": int(self.estimator.predict(x)[0])}


# # Model Register

In [ ]:
# Package this class + `src/` (including weights.pkl from train.ipynb)
# as a model in the MLflow Registry. Returns a `models:/<name>/<version>` URI.
#   first arg     — NubisonModel implementation
#   artifact_dirs — comma-separated dirs to ship with the model
#   tags          — model-version tags. mlplatform reads
#                   `inference_server_image` to choose the serving image.
from nubison_model import register

model_id = register(
    UserModel(),
    artifact_dirs="src",
    tags={"inference_server_image": "ghcr.io/nubison/nubison-model:0.0.7"},
)
print(f"Model registered: {model_id}")


# # Model Testing

In [ ]:
# Run the model locally over HTTP for a smoke test.
#   test_client(model_id) yields a TestClient — use .post("/infer", json=...)
from nubison_model.Service import test_client

with test_client(model_id) as client:
    result = client.post(
        "/infer",
        json={
            "sepal_length": 5.1, "sepal_width": 3.5,
            "petal_length": 1.4, "petal_width": 0.2,
        },
    )
    print("predicted target:", result.json()["target"])
